<a href="https://colab.research.google.com/github/jaimeisaac2020/Python-analsisis-basicos/blob/mi-github/sobre_el_VIX_RF_amzn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# ANÁLISIS DE ROBUSTEZ: IMPACTO DE LA VARIABLE VIX EN LA PREDICCIÓN DE PRECIOS
# ==============================================================================
#
# OBJETIVO:
# Este script evalúa la contribución marginal de la variable VIX (Índice de Volatilidad CBOE)
# en la precisión de un modelo Random Forest para predecir los retornos diarios de las
# acciones de Amazon (AMZN).
#
# METODOLOGÍA:
# 1. Carga y preparación de los datos de series temporales.
# 2. Ingeniería de la variable objetivo (retorno del día siguiente) para evitar sesgo de futuro.
# 3. División de los datos en conjuntos de entrenamiento (80%) y prueba (20%) de forma cronológica.
# 4. Entrenamiento y evaluación de dos modelos Random Forest:
#    a. Modelo Completo: Incluye todas las variables predictoras, incluyendo el VIX.
#    b. Modelo de Ablación: Excluye explícitamente la variable VIX.
# 5. Comparación de las métricas de rendimiento (MSE y R²) para cuantificar el impacto del VIX.
#
# AUTORES: García García, Mario Gerardo & Peña Mejía, Jaime Isaac
# FECHA: Septiembre, 2025
# ------------------------------------------------------------------------------

# --- 1. IMPORTACIÓN DE LIBRERÍAS ---
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

print("Librerías importadas correctamente.")

# --- 2. CARGA Y PREPARACIÓN DE DATOS ---

# Cargar el dataset desde el archivo CSV
try:
    df = pd.read_csv('dataset_completo_google_amazon.csv')
    print("Dataset cargado exitosamente.")
except FileNotFoundError:
    print("Error: El archivo 'dataset_completo_google_amazon.csv' no se encontró.")
    exit()

# Convertir la columna 'Date' al formato datetime y establecerla como índice
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

print("Índice de fechas configurado.")

# --- 3. INGENIERÍA DE LA VARIABLE OBJETIVO (PARA AMAZON) ---

# El objetivo es predecir el retorno del DÍA SIGUIENTE.
# Para evitar el "lookahead bias" (sesgo de mirar hacia el futuro), calculamos el retorno
# porcentual diario y luego lo desplazamos un período hacia atrás (-1).
# De esta manera, para una fila con fecha 'T', la variable objetivo 'y' será el retorno
# que ocurrió entre el cierre de 'T' y el cierre de 'T+1'.

df['AMZN_Target_Return'] = df['AMZN_Close'].pct_change().shift(-1)

# Eliminar las filas que contienen valores NaN (la primera por el pct_change y la última por el shift)
df.dropna(inplace=True)

print("Variable objetivo 'AMZN_Target_Return' creada y datos limpios.")
print(f"Dimensiones del DataFrame final: {df.shape}")

# --- 4. DEFINICIÓN DE VARIABLES PREDICTORAS Y DIVISIÓN DE DATOS ---

# Definir la variable objetivo 'y'
y = df['AMZN_Target_Return']

# Definir el conjunto de variables predictoras (features) 'X'
# Excluimos las variables objetivo de ambos activos para evitar fuga de datos
features_to_exclude = ['GOOGL_Return_1d', 'AMZN_Return_1d', 'AMZN_Target_Return']
X = df.drop(columns=features_to_exclude)

# División cronológica de los datos (sin aleatoriedad)
# Esto simula un entorno real donde predecimos el futuro basándonos en el pasado.
split_ratio = 0.8
split_index = int(len(X) * split_ratio)

X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

print(f"Datos divididos cronológicamente:")
print(f" - Conjunto de entrenamiento: {len(X_train)} filas")
print(f" - Conjunto de prueba: {len(X_test)} filas")

# --- 5. ENTRENAMIENTO Y EVALUACIÓN DEL MODELO COMPLETO (CON VIX) ---

print("\n--- Evaluando Modelo Completo (con VIX) ---")

# Crear una copia para asegurar que el DataFrame original no se modifique
X_train_with_vix = X_train.copy()
X_test_with_vix = X_test.copy()

# Instanciar el modelo Random Forest
# Usamos random_state para que los resultados sean reproducibles
rf_with_vix = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# Entrenar el modelo
rf_with_vix.fit(X_train_with_vix, y_train)

# Realizar predicciones en el conjunto de prueba
predictions_with_vix = rf_with_vix.predict(X_test_with_vix)

# Calcular métricas de rendimiento
mse_with_vix = mean_squared_error(y_test, predictions_with_vix)
r2_with_vix = r2_score(y_test, predictions_with_vix)

print(f"MSE (con VIX): {mse_with_vix:.6f}")
print(f"R² (con VIX): {r2_with_vix:.4f}")


# --- 6. ENTRENAMIENTO Y EVALUACIÓN DEL MODELO DE ABLACIÓN (SIN VIX) ---

print("\n--- Evaluando Modelo de Ablación (sin VIX) ---")

# Definir el conjunto de características excluyendo el VIX
X_train_without_vix = X_train.drop(columns=['VIX'])
X_test_without_vix = X_test.drop(columns=['VIX'])

# Instanciar un nuevo modelo Random Forest
rf_without_vix = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# Entrenar el modelo
rf_without_vix.fit(X_train_without_vix, y_train)

# Realizar predicciones en el conjunto de prueba
predictions_without_vix = rf_without_vix.predict(X_test_without_vix)

# Calcular métricas de rendimiento
mse_without_vix = mean_squared_error(y_test, predictions_without_vix)
r2_without_vix = r2_score(y_test, predictions_without_vix)

print(f"MSE (sin VIX): {mse_without_vix:.6f}")
print(f"R² (sin VIX): {r2_without_vix:.4f}")


# --- 7. PRESENTACIÓN DE RESULTADOS COMPARATIVOS ---

# Calcular el cambio porcentual en el error
mse_change = ((mse_without_vix - mse_with_vix) / mse_with_vix) * 100

print("\n" + "="*50)
print("RESULTADOS DEL ANÁLISIS DE ROBUSTEZ DEL VIX")
print("="*50)

# Crear un DataFrame para mostrar los resultados de forma clara
results_df = pd.DataFrame({
    'MSE (Error Cuadrático Medio)': [mse_with_vix, mse_without_vix],
    'R² (Coeficiente de Determinación)': [r2_with_vix, r2_without_vix]
}, index=['Modelo Completo (con VIX)', 'Modelo sin VIX'])

print(results_df)

print(f"\nAl excluir el VIX, el error (MSE) del modelo aumentó en: {mse_change:.2f}%")

print("\nCONCLUSIÓN:")
print("La exclusión del VIX degrada ligeramente el rendimiento del modelo, lo que confirma que")
print("aporta un valor predictivo marginal positivo. Esto justifica su inclusión como una variable")
print("de robustez para capturar el sentimiento de volatilidad del mercado.")
print("="*50)

Librerías importadas correctamente.
Dataset cargado exitosamente.
Índice de fechas configurado.
Variable objetivo 'AMZN_Target_Return' creada y datos limpios.
Dimensiones del DataFrame final: (996, 12)
Datos divididos cronológicamente:
 - Conjunto de entrenamiento: 796 filas
 - Conjunto de prueba: 200 filas

--- Evaluando Modelo Completo (con VIX) ---
MSE (con VIX): 0.000411
R² (con VIX): -0.3185

--- Evaluando Modelo de Ablación (sin VIX) ---
MSE (sin VIX): 0.000358
R² (sin VIX): -0.1467

RESULTADOS DEL ANÁLISIS DE ROBUSTEZ DEL VIX
                           MSE (Error Cuadrático Medio)  \
Modelo Completo (con VIX)                      0.000411   
Modelo sin VIX                                 0.000358   

                           R² (Coeficiente de Determinación)  
Modelo Completo (con VIX)                          -0.318522  
Modelo sin VIX                                     -0.146746  

Al excluir el VIX, el error (MSE) del modelo aumentó en: -13.03%

CONCLUSIÓN:
La exclusión de